In [1]:
%matplotlib inline
import sys
sys.path.append("/global/cfs/projectdirs/m4491/btaczak/SIBIL-runs/helper_scripts")

In [2]:
# =============================================================================
# Simple interactive radial profile plotter (Jupyter / NERSC)
# Replaces mapper.bout / mapper.solps indexing with plain numpy.load arrays.
# Run once; use the slider to scan through poloidal index (like `jomp`).
# =============================================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed
from boutdata.data import BoutData
from boututils.datafile import DataFile
from converged_profile_functions import unbiased_convergence, end_biased_convergence

In [3]:
# -----------------------------------------------------------------------------
# 1) LOAD DATA -- EDIT PATHS
# -----------------------------------------------------------------------------
bout_dir  = Path(f"/global/cfs/projectdirs/m4491/btaczak/SIBIL-runs/testing/0_iteration/Shelf_new_grid2_sinks2_gradfix")
data_dir  = bout_dir / "numpy"
grid_fname= bout_dir / "d3d_shelf_184016.03200_nc3.grd.nc"
grid_fname = str(grid_fname)
d = BoutData(path=bout_dir)

grid = DataFile(grid_fname)
try:
    _ = grid["psi_bdry"]
    has_psi_bdry = True
except Exception:
    has_psi_bdry = False
    
if has_psi_bdry:
    psi_bndry = grid["psi_bdry"]
else:
    psi_bndry = grid["psi_bndry"]

jomp = 44
psi_midplane = grid["psixy"][:,jomp]
psin_midplane = ((grid["psixy"] - grid["psi_axis"]) / (psi_bndry - grid["psi_axis"]))[:,jomp]

psin  = psin_midplane   # (n_radial, n_pol)
D     = np.load(f"{data_dir}/Da_t.npy")       # (t, n_radial, n_pol)  -- particle diffusivity
Xe = np.load(f"{data_dir}/Xe_t.npy")          # (t, n_radial, n_pol)  -- electron heat diff.
Xi = np.load(f"{data_dir}/Xi_t.npy")          # (t, n_radial, n_pol)  -- ion heat diff.

/global/homes/b/btaczak/.conda/envs/bout-data/lib/python3.13/site-packages/boutdata/data.py:765: AlwaysWarning: While building x, y, z coordinate arrays, an exception occured: 'list' object has no attribute 'split'
Evaluating non-scalar options not available
  alwayswarn(


In [9]:
arrays = {"D": D, "Xe": Xe, "Xi": Xi}

In [11]:
# -----------------------------------------------------------------------------
# 2) AUTO-DETECT SHAPES
# -----------------------------------------------------------------------------
# Poloidal size = last axis of any array
n_pol = D.shape[-1]

# Does anything (including psin) carry a time axis?
has_time = any(a.ndim == 3 for a in list(arrays.values()) + [psin])
n_time = None
if has_time:
    time_arrays = [a for a in list(arrays.values()) + [psin] if a.ndim == 3]
    n_time = time_arrays[0].shape[0]
    for a in time_arrays:
        assert a.shape[0] == n_time, "Mismatched n_time across arrays"


def get_slice(arr, j, t):
    """Return the 1D radial profile at poloidal index j (and time t if present)."""
    if arr.ndim == 3:
        return arr[t, :, j]
    else:
        return arr[:, j]

print(f"Timesteps: {n_time}")

Timesteps: 196


In [39]:
N,l = 100, 196
D_conv = unbiased_convergence(D,N,l)
Xe_conv = unbiased_convergence(Xe,N,l)
Xi_conv = unbiased_convergence(Xi,N,l)

# D_conv = end_biased_convergence(D,N,l)
# Xe_conv = end_biased_convergence(Xe,N,l)
# Xi_conv = end_biased_convergence(Xi,N,l)

In [32]:
arrays = {"D": D_conv, "Xe": Xe_conv, "Xi": Xi_conv}

In [33]:
# -----------------------------------------------------------------------------
# 3) PLOT FUNCTION -- creates a brand-new figure each call (inline-friendly)
# -----------------------------------------------------------------------------
colors_conv = {"D_conv": "C0", "Xe_conv": "C1", "Xi_conv": "C2"}
colors = {"D": "C0", "Xe": "C1", "Xi": "C2"}
labels = {"D": r"$D$", "Xe": r"$\chi_e$", "Xi": r"$\chi_i$"}

def plot_profiles(j, t=0):
    x = psin
    fig, axs = plt.subplots(3, 1, figsize=(4, 5), sharex=True)

    for ax, name in zip(axs, arrays.keys()):
        y = get_slice(arrays[name], j, t)
        ax.plot(x, y, color=colors[name], label=labels[name])
        ax.set_ylabel(labels[name])
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="x", labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.yaxis.get_offset_text().set_fontsize(8)

    axs[2].set_xlabel(r"$\psi_n$")
    title = f"Poloidal index = {j}"
    if has_time:
        title += f"   |   Time index = {t}"
    #fig.suptitle(title)
    # fig.tight_layout()
    plt.show()

# -----------------------------------------------------------------------------
# 4) WIRE UP SLIDERS -- run this cell once
# -----------------------------------------------------------------------------
if has_time:
    interact(
        plot_profiles,
        j=IntSlider(min=0, max=n_pol - 1, step=1, value=0, description="Pol. idx:"),
        #t=IntSlider(min=0, max=n_time - 1, step=1, value=0, description="Time idx:"),
        t=IntSlider(min=0, max=l - 1, step=1, value=0, description="Time idx:"),
    )
else:
    interact(
        plot_profiles,
        j=IntSlider(min=0, max=n_pol - 1, step=1, value=0, description="Pol. idx:"),
        t=fixed(0),
    )

interactive(children=(IntSlider(value=0, description='Pol. idx:', max=63), IntSlider(value=0, description='Tim…